# pyld 2.x Term Selection Behavior

Demonstrates term selection in `jsonld.compact()` when multiple context terms map to the same IRI.

- **pyld 2.x** sorts terms lexicographically first, then by length (non-spec-compliant).
- **pyld 3.0** sorts by length first, then lexicographically (spec-compliant per W3C JSON-LD 1.1 API).

The difference manifests when the shorter term sorts lexicographically AFTER the longer term.

Reference: https://github.com/digitalbazaar/pyld/issues/247

Use case: [OO-LD (Object Oriented Linked Data)](https://github.com/OO-LD) / [OpenSemanticLab](https://github.com/OpenSemanticLab)

In [ ]:
# Install pyld 2.x (pre-3.0, non-spec-compliant term ordering)
# For JupyterLite: this uses micropip under the hood
%pip install "pyld==2.0.4"

In [ ]:
from pyld import jsonld
import json

# Verify version
import importlib.metadata
version = importlib.metadata.version("pyld")
print(f"pyld version: {version}")


def run_scenario(title, context, doc, v2_expected, v3_expected):
    """Compact and report which term was selected."""
    result = jsonld.compact(doc, context)
    terms = {k: v for k, v in result.items() if not k.startswith("@")}
    selected = list(terms.keys())

    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    print(f"Result: {json.dumps(terms, indent=2)}")
    print(f"Selected term(s): {selected}")
    print(f"  pyld 2.x would select: '{v2_expected}'")
    print(f"  pyld 3.0 would select: '{v3_expected}'")

    return result, selected

In [ ]:
# Scenario 1: Simple duplicate IRI, different lengths
# 'full_name' (9 chars, starts 'f') vs 'name' (4 chars, starts 'n')
# Both map to https://schema.org/name
#
# pyld 2.x: lex first -> 'f' < 'n' -> selects 'full_name' (the LONGER term)
# pyld 3.0: length first -> 4 < 9 -> selects 'name' (the SHORTER term)

context = {
    "schema": "https://schema.org/",
    "name": "schema:name",
    "full_name": "schema:name",
}
doc = {"https://schema.org/name": [{"@value": "Alice"}]}

result, selected = run_scenario(
    "Scenario 1: name (4) vs full_name (9) - same IRI",
    context, doc,
    v2_expected="full_name",
    v3_expected="name",
)
assert "full_name" in selected, f"pyld 2.x should select 'full_name', got {selected}"
print("PASS: pyld 2.x selects 'full_name' (lexicographically first: 'f' < 'n')")

In [ ]:
# Scenario 2 (Control): Same-length terms
# 'name' (4 chars) vs 'nick' (4 chars) -> both map to schema:name
# Same length -> lex tiebreak in BOTH versions -> 'name' < 'nick'

context = {
    "schema": "https://schema.org/",
    "name": "schema:name",
    "nick": "schema:name",
}
doc = {"https://schema.org/name": [{"@value": "Alice"}]}

result, selected = run_scenario(
    "Scenario 2 (Control): name (4) vs nick (4) - same length",
    context, doc,
    v2_expected="name",
    v3_expected="name",
)
assert "name" in selected, f"Both versions should select 'name', got {selected}"
print("PASS: Same in both versions (same length, lex tiebreak)")

In [ ]:
# Scenario 3: OO-LD transform intermediate context
# During jsonld_to_jsonld() in oold-python, temporary terms like '_demo_full_name'
# are created alongside 'name', both mapping to the same IRI (schema:name).
# This happens in the flatten step (transform.py line 93-94).
#
# '_demo_full_name' (15 chars, starts '_') vs 'name' (4 chars, starts 'n')
# pyld 2.x: lex first -> '_' (0x5F) < 'n' (0x6E) -> selects '_demo_full_name'
# pyld 3.0: length first -> 4 < 15 -> selects 'name'

context = {
    "schema": "http://schema.org/",
    "demo": "https://oo-ld.github.io/demo/",
    "name": "schema:name",
    "_demo_full_name": {"@id": "schema:name"},
    "type": "@type",
    "id": "@id",
}
doc = {
    "@type": ["http://schema.org/Person"],
    "http://schema.org/name": [{"@value": "Alice"}],
}

result, selected = run_scenario(
    "Scenario 3: OO-LD intermediate - name (4) vs _demo_full_name (15)",
    context, doc,
    v2_expected="_demo_full_name",
    v3_expected="name",
)
assert "_demo_full_name" in selected, f"pyld 2.x should select '_demo_full_name', got {selected}"
print("PASS: pyld 2.x selects '_demo_full_name' (underscore sorts before 'n')")

In [ ]:
# Scenario 4 (Control): Real OSW Entity schema
# 'close_ontology_match*' (21 chars) and 'exact_ontology_match*' (21 chars)
# Both map to Property:Equivalent_URI in the Entity schema.
# Same length -> lex tiebreak in both versions -> 'c' < 'e'

context = {
    "wiki": "https://wiki.example.org/id/",
    "Property": {"@id": "wiki:Property-3A", "@prefix": True},
    "close_ontology_match*": {"@id": "Property:Equivalent_URI", "@type": "@id"},
    "exact_ontology_match*": {"@id": "Property:Equivalent_URI", "@type": "@id"},
}
doc = {
    "https://wiki.example.org/id/Property-3AEquivalent_URI": [
        {"@id": "https://example.org/SomeOntologyClass"}
    ]
}

result, selected = run_scenario(
    "Scenario 4 (Control): OSW Entity - close_ontology_match* (21) vs exact_ontology_match* (21)",
    context, doc,
    v2_expected="close_ontology_match*",
    v3_expected="close_ontology_match*",
)
assert "close_ontology_match*" in selected, f"Both should select 'close_ontology_match*', got {selected}"
print("PASS: Same in both versions (same length, lex tiebreak: 'c' < 'e')")

In [ ]:
# Scenario 5: Full OO-LD jsonld_to_jsonld transform
# The OO-LD transform pipeline uses three steps:
#   1. compact (with temp terms nulling out * aliases)
#   2. flatten (with combined context that HAS same-IRI collisions)
#   3. compact (with clean original context, NO collisions)
#
# The intermediate flatten step (2) selects different terms in v2 vs v3,
# but the final compact (3) normalizes via expand/compact,
# so the final output is IDENTICAL in both versions.

def jsonld_to_jsonld_simplified(graph, transformation_context):
    """Simplified from oold-python/src/oold/utils/transform.py"""
    temp1 = {}
    temp2 = {}

    for key, value in transformation_context.items():
        if key.endswith("*"):
            temp1_value = {}
            temp2_value = {}
            if isinstance(value, dict):
                if "@id" in value:
                    temp1_value["@id"] = value["@id"]
                if "@reverse" in value:
                    temp1_value["@id"] = value["@reverse"]
                if "@type" in value:
                    temp1_value["@type"] = value["@type"]
                temp2_value = {**value}
            else:
                temp1_value["@id"] = value
                temp2_value["@id"] = value

            org_key = key.replace("*", "")
            org_value = transformation_context[org_key]
            if isinstance(org_value, dict):
                if "@id" in org_value:
                    if "@id" in temp2_value:
                        temp2_value["@id"] = org_value["@id"]
                    if "@reverse" in temp2_value:
                        temp2_value["@reverse"] = org_value["@id"]
            else:
                if "@id" in temp2_value:
                    temp2_value["@id"] = org_value
                if "@reverse" in temp2_value:
                    temp2_value["@reverse"] = org_value

            temp1["_" + temp1_value["@id"].replace(":", "_")] = temp1_value
            temp2["_" + temp1_value["@id"].replace(":", "_")] = temp2_value
            temp1[key] = None

    graph = jsonld.compact(graph, {**transformation_context, **temp1})
    graph["@context"] = {**transformation_context, **temp2}
    graph = jsonld.flatten(graph)
    graph = jsonld.compact(graph, transformation_context)
    return graph


graph = {
    "@context": {
        "schema": "http://schema.org/",
        "demo": "https://oo-ld.github.io/demo/",
        "name": "schema:name",
        "full_name": "demo:full_name",
        "label": "demo:label",
        "works_for": {"@id": "schema:worksFor", "@type": "@id"},
        "is_employed_by": {"@id": "demo:is_employed_by", "@type": "@id"},
        "employes": {"@id": "schema:employes", "@type": "@id"},
        "type": "@type",
        "id": "@id",
    },
    "@graph": [
        {"id": "demo:person1", "type": "schema:Person", "name": "Person1",
         "works_for": "demo:organizationA"},
        {"id": "demo:person2", "type": "schema:Person", "full_name": "Person2",
         "is_employed_by": "demo:organizationA"},
        {"id": "demo:person3", "type": "schema:Person", "name": "Person3"},
        {"id": "demo:organizationA", "type": "schema:Organization",
         "label": "organizationA", "employes": "demo:person3"},
    ],
}

transform_context = {
    "schema": "http://schema.org/",
    "demo": "https://oo-ld.github.io/demo/",
    "skos": "http://www.w3.org/2004/02/skos/core#",
    "name": "schema:name",
    "name*": "demo:full_name",
    "text": "@value",
    "lang": "@language",
    "label": {"@id": "skos:prefLabel", "@container": "@set"},
    "label*": {"@id": "demo:label", "@container": "@set", "@language": "en"},
    "employes": {"@id": "schema:employes", "@type": "@id"},
    "employes*": {"@reverse": "schema:worksFor", "@type": "@id"},
    "employes**": {"@reverse": "demo:is_employed_by", "@type": "@id"},
    "type": "@type",
    "id": "@id",
}

result = jsonld_to_jsonld_simplified(graph, transform_context)

print(f"\n{'='*60}")
print("Scenario 5: Full OO-LD jsonld_to_jsonld transform")
print(f"{'='*60}")
print(json.dumps(result, indent=2))

expected_graph = [
    {
        "employes": ["demo:person1", "demo:person2", "demo:person3"],
        "id": "demo:organizationA",
        "label": [{"lang": "en", "text": "organizationA"}],
        "type": "schema:Organization",
    },
    {"id": "demo:person1", "name": "Person1", "type": "schema:Person"},
    {"id": "demo:person2", "name": "Person2", "type": "schema:Person"},
    {"id": "demo:person3", "name": "Person3", "type": "schema:Person"},
]

assert result["@graph"] == expected_graph, (
    f"Transform output differs!\n"
    f"Expected:\n{json.dumps(expected_graph, indent=2)}\n"
    f"Got:\n{json.dumps(result.get('@graph'), indent=2)}"
)
print("\nPASS: Final output identical in both versions")
print("(The expand/compact pipeline normalizes intermediate term selection differences)")

In [ ]:
# Scenario 6: import_jsonld pattern
# When deserializing expanded JSON-LD into Pydantic models via oold-python's
# import_jsonld(), compact() uses the model's context. If two terms compete
# for the same IRI, the selected term determines the dict key, which MUST
# match the Pydantic field name for successful deserialization.
#
# 'display_name' (12 chars, starts 'd') vs 'name' (4 chars, starts 'n')
# pyld 2.x: lex first -> 'd' < 'n' -> selects 'display_name'
# pyld 3.0: length first -> 4 < 12 -> selects 'name'

context = {
    "schema": "https://schema.org/",
    "ex": "https://example.org/",
    "id": "@id",
    "type": "@type",
    "name": "schema:name",
    "display_name": "schema:name",
}
expanded = {
    "@id": "https://example.org/alice",
    "@type": ["https://example.org/Person"],
    "https://schema.org/name": [{"@value": "Alice"}],
}

result, selected = run_scenario(
    "Scenario 6: import_jsonld - name (4) vs display_name (12)",
    context, expanded,
    v2_expected="display_name",
    v3_expected="name",
)
assert "display_name" in selected, f"pyld 2.x should select 'display_name', got {selected}"
print("PASS: pyld 2.x selects 'display_name' (lexicographically first: 'd' < 'n')")
print("\nIMPACT: If a Pydantic model has field 'name' but compact() returns")
print("'display_name', deserialization fails. This is the main risk for import_jsonld.")

## Summary

| Scenario | Terms | Same IRI | pyld 2.x | pyld 3.0 | Differs? |
|----------|-------|----------|----------|----------|----------|
| 1. Simple duplicate | `name` (4) vs `full_name` (9) | schema:name | `full_name` | `name` | Yes |
| 2. Same length (control) | `name` (4) vs `nick` (4) | schema:name | `name` | `name` | No |
| 3. OO-LD intermediate | `name` (4) vs `_demo_full_name` (15) | schema:name | `_demo_full_name` | `name` | Yes |
| 4. OSW Entity (control) | `close_ontology_match*` (21) vs `exact_ontology_match*` (21) | Property:Equivalent_URI | `close_ontology_match*` | `close_ontology_match*` | No |
| 5. Full transform | N/A (pipeline) | N/A | identical output | identical output | No |
| 6. import_jsonld | `name` (4) vs `display_name` (12) | schema:name | `display_name` | `name` | Yes |

**Key finding**: The OO-LD `jsonld_to_jsonld` transform pipeline is resilient because the final `compact()` uses a clean context without same-IRI collisions. However, `import_jsonld()` and any direct `compact()` call with duplicate-IRI contexts is affected.